<a href="https://colab.research.google.com/github/yoorahazel-ctrl/Spam_Email_Classifier/blob/main/Spam_Email_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sklearn

print("Success! Your AI environment is ready to go.")

Success! Your AI environment is ready to go.


In [ ]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

# Load a built-in dataset of real emails
newsgroups = fetch_20newsgroups(subset='all', categories=['comp.graphics', 'rec.sport.baseball'], remove=('headers', 'footers', 'quotes'))

# Convert it into our spreadsheet format (DataFrame)
df = pd.DataFrame({'message': newsgroups.data, 'label': newsgroups.target})

# To make it beginner-friendly, let's map the numbers to actual text labels:
# 0 will mean 'computer graphics' (our fake "ham") and 1 will mean 'baseball' (our fake "spam")
df['label'] = df['label'].map({0: 'ham', 1: 'spam'})

# Drop any empty rows just in case
df = df.dropna().reset_index(drop=True)

# Look at the first 5 rows
df.head()

,message,label
0,: Are there significant differences between V2...,ham
1,"\nOops, forgot the phone number. It's 1-800-3...",ham
2,I'd love to see a Shea Stadium gif.\n\n\n ...,spam
3,\n The event that had the most impact on Gaet...,spam
4,"\n\n\nIt's not a cliche, and (unlike your comm...",spam


In [ ]:
# 1. Count how many total rows (emails) we have
print("Total number of messages:", len(df))

# 2. Count how many are 'ham' vs 'spam'
print("\nHow many of each label:")
print(df['label'].value_counts())

# 3. Look at one actual email message fully to see what the text looks like
print("\nExample of a single message:")
print(df['message'].iloc[0])

Total number of messages: 1967

How many of each label:
label
spam    994
ham     973
Name: count, dtype: int64

Example of a single message:
: Are there significant differences between V2.01 and V2.00 ?
: Thank you for helping


No.  As I recall, the only differences are in the 3ds.set parameters - some
of the defaults have changed slightly.  I'll look when I get home and let
you know, but there isn't enough to actually warrant upgrading.


In [ ]:
# A function to clean our text
def clean_text(text):
    # Make text lowercase
    text = text.lower()

    # Remove punctuation characters like !, ., ?, etc.
    punctuation_chars = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
    for char in punctuation_chars:
        text = text.replace(char, '')

    # Remove extra spaces and line breaks
    text = " ".join(text.split())
    return text

# Apply this cleaning function to our entire dataset
df['cleaned_message'] = df['message'].apply(clean_text)

# Look at the original vs the cleaned text side-by-side
df[['message', 'cleaned_message']].head()

,message,cleaned_message
0,: Are there significant differences between V2...,are there significant differences between v201...
1,"\nOops, forgot the phone number. It's 1-800-3...",oops forgot the phone number its 18003778287
2,I'd love to see a Shea Stadium gif.\n\n\n ...,id love to see a shea stadium gif sean
3,\n The event that had the most impact on Gaet...,the event that had the most impact on gaettis ...
4,"\n\n\nIt's not a cliche, and (unlike your comm...",its not a cliche and unlike your comments belo...


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# 1. Split our data into 80% for training and 20% for testing
# X is our text data, y is our labels (ham/spam)
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_message'],
    df['label'],
    test_size=0.20,
    random_state=42 # This number ensures we get the exact same split if we run it again
)

# 2. Initialize the vectorizer tool
vectorizer = CountVectorizer()

# 3. Feed the training text to the vectorizer so it learns all the unique words
X_train_numbers = vectorizer.fit_transform(X_train)

# 4. Convert the test text into numbers using that same vocabulary
X_test_numbers = vectorizer.transform(X_test)

print("Data successfully split and converted!")
print(f"Training samples: {X_train_numbers.shape[0]} | Testing samples: {X_test_numbers.shape[0]}")

Data successfully split and converted!
Training samples: 1573 | Testing samples: 394


In [ ]:
from sklearn.naive_bayes import MultinomialNB

# 1. Initialize our AI model
ai_classifier = MultinomialNB()

# 2. Train the model by feeding it the training numbers and the correct answers (labels)
ai_classifier.fit(X_train_numbers, y_train)

print("Congratulations! Your AI model has been successfully trained.")

Congratulations! Your AI model has been successfully trained.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Ask the AI to predict the labels for our test data
predictions = ai_classifier.predict(X_test_numbers)

# 2. Calculate the overall accuracy score
accuracy = accuracy_score(y_test, predictions)
print(f"--- Final Exam Results ---")
print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

# 3. Print a detailed scorecard (Precision and Recall)
print("Detailed Scorecard:")
print(classification_report(y_test, predictions))

--- Final Exam Results ---
Overall Accuracy: 95.43%

Detailed Scorecard:
              precision    recall  f1-score   support

         ham       0.97      0.94      0.95       203
        spam       0.93      0.97      0.95       191

    accuracy                           0.95       394
   macro avg       0.95      0.95      0.95       394
weighted avg       0.96      0.95      0.95       394



In [ ]:
def test_my_own_message(custom_message):
    # 1. Clean the custom text just like we cleaned the dataset
    cleaned_custom = clean_text(custom_message)

    # 2. Turn the custom text into numbers using our existing vectorizer
    custom_numbers = vectorizer.transform([cleaned_custom])

    # 3. Let the AI predict!
    prediction = ai_classifier.predict(custom_numbers)[0]

    print(f"Message: '{custom_message}'")
    print(f"AI Prediction: This message is... {prediction.upper()}!\n")

# --- TRY IT OUT HERE ---
# Feel free to change these sentences to whatever you want and re-run the cell!
test_my_own_message("Hey, are we still playing baseball this afternoon?")
test_my_own_message("Download beautiful high resolution 3D computer graphics for your desktop now!")
test_my_own_message("The pitcher threw a perfect fastball for a strikeout.")
test_my_own_message("We need to render the 3D polygon mesh using Blender software.")

Message: 'Hey, are we still playing baseball this afternoon?'
AI Prediction: This message is... SPAM!

Message: 'Download beautiful high resolution 3D computer graphics for your desktop now!'
AI Prediction: This message is... HAM!

Message: 'The pitcher threw a perfect fastball for a strikeout.'
AI Prediction: This message is... SPAM!

Message: 'We need to render the 3D polygon mesh using Blender software.'
AI Prediction: This message is... HAM!

